In [1]:
import fitz
import base64
import json
import pandas as pd
from mistralai.client import Mistral

def extract_image_from_pdf(pdf_path, page_number=0, zoom=3):
    """Step 1: Extract image from PDF"""
    doc = fitz.open(pdf_path)
    page = doc[page_number]
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    image_path = "temp_invoice.png"
    pix.save(image_path)
    print(f"✅ Image extracted from PDF")
    return image_path

def convert_image_to_base64(image_path):
    """Step 2: Convert image to base64"""
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    print(f"✅ Image converted to base64")
    return base64_image

def extract_invoice_data(base64_image, api_key):
    """Step 3: Send to Mistral and extract data"""
    client = Mistral(api_key=api_key)
    response = client.chat.complete(
        model="pixtral-12b-2409",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Extract ONLY visible data from this invoice. Respond ONLY in JSON: {\"invoice_number\": \"\", \"date\": \"\", \"supplier\": \"\", \"total_ht\": 0.0, \"total_tva\": 0.0, \"total_ttc\": 0.0}"
                    },
                    {
                        "type": "image_url",
                        "image_url": f"data:image/png;base64,{base64_image}"
                    }
                ]
            }
        ]
    )
    raw = response.choices[0].message.content
    clean = raw.replace("```json", "").replace("```", "").strip()
    data = json.loads(clean)
    print(f"✅ Data extracted: {data}")
    return data

def save_to_excel(data, output_path="invoices_output.xlsx"):
    """Step 4: Save to Excel"""
    try:
        existing_df = pd.read_excel(output_path)
        new_df = pd.DataFrame([data])
        final_df = pd.concat([existing_df, new_df], ignore_index=True)
    except FileNotFoundError:
        final_df = pd.DataFrame([data])
    
    final_df.to_excel(output_path, index=False)
    print(f"✅ Saved to Excel! Total invoices: {len(final_df)}")
    return final_df

print("✅ All functions loaded!")


✅ All functions loaded!


In [2]:
# Run the full pipeline
API_KEY = "s46f3EZ1Up9LrY0INTDQ8wyGMSYggC05"

# Step 1: Extract image from PDF
image_path = extract_image_from_pdf("invoice.pdf")

# Step 2: Convert to base64
base64_image = convert_image_to_base64(image_path)

# Step 3: Extract data with Mistral
data = extract_invoice_data(base64_image, API_KEY)

# Step 4: Save to Excel
df = save_to_excel(data)

print("\n🎉 Pipeline complete!")
print(df)

✅ Image extracted from PDF
✅ Image converted to base64
✅ Data extracted: {'invoice_number': '1253888565', 'date': '27.04.2026', 'supplier': 'SYSCO France SAS', 'total_ht': 526.47, 'total_tva': 28.96, 'total_ttc': 555.43}
✅ Saved to Excel! Total invoices: 4

🎉 Pipeline complete!
  invoice_number        date          supplier  total_ht  total_tva  total_ttc
0     1253888565  27.04.2026  SYSCO France SAS    526.47      28.96     555.43
1     1253642565  02.04.2026  SYSCO France SAS    234.20      12.88     247.08
2     1253743874  13.04.2026  SYSCO France SAS    424.39      30.10     454.49
3     1253888565  27.04.2026  SYSCO France SAS    526.47      28.96     555.43
